# Chronic Kidney Diesease Inclusion Criteria Search

## Inclusion Criteria:
Adult Patients who have had:
1. GFR Measurement < 60
2. Subsequent GFR Measurement < 60 between 3 & 12 Months after the first
3. Further GFR Measurement < 70 at least 12 Months after the second measurement

In [ ]:
import sys
sys.path.append('..')
from credentials import *

from elasticsearch_utils import *

import duckdb
import pandas as pd
import numpy as np
import re

data_path = "../data/"
raw_data_path = data_path+'raw_data/'

In [ ]:
# --- Connect to Elasticsearch ---
es = connect_elasticsearch(hosts=hosts, username=username, password=password, api_key=True)

## Extract all GFR Measurementss
### Only measurements <70 extracted, as those are only results of relevance
This dataframe, *measure_gfrs_df*, is the base dataframe for all inclusion criteria patient extraction

In [ ]:
with duckdb.connect() as conn:
    omop_measures_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                                       SELECT DISTINCT m.master_person_id
                                              , p.birth_datetime
                                              , m.measurement_date
                                              , 'eGFR' AS measurement_source_value_name
                                              , m.value_as_number
                                              , 'mL/min', unit_source_value
                                        FROM ext_measurement as m
                                            LEFT JOIN ext_person as p
                                                USING(master_person_id)
                                        WHERE UPPER(m.measurement_source_value_name) LIKE '%GFR%' AND m.value_as_number IS NOT NULL
                                                                                                  AND m.value_as_number < 70
                                        ;""").df()

In [ ]:
# --- Explore Fields in a Given Index ---
index = 'lab_results'
columns = ['patient_identifier1', 'patient_activity_document_identifiers', 'patient_identifier2', 'patient_identifier3',
           'patient_EpicId', 'document_Name', 'document_CreatedWhen', 'document_Fields']

In [ ]:
gfr_query = {
    "from": 0,
    "size": 10000,
    "query": {
        "bool": {
            "filter": [
                {
                    "range": {
                        "activity_Date": {
                            "gte": "2023-10-01T00:00:00.000+01:00"
                            }
                        }
                    },
                {
                    "nested": {
                        "path": "document_Fields",
                        "query": {
                            "bool": {
                                "should": [
                                    {
                                        "terms": {
                                            "document_Fields.label": [
                                                "eGFR",
                                                "eGFR by CKD-EPI (2009)",
                                                "eGFR by MDRD",
                                                "EGFR BY CKD -15 MINUTES"
                                                ]
                                            }
                                        }
                                    ],
                                "minimum_should_match": 1
                                }
                            },
                        "score_mode": "none"
                        }
                    }
                ]
            }
        }
    }

In [ ]:
epic_egfr_df = es_docs_to_df(es, index=index, query=gfr_query, column_headers=columns, timeout=600)

In [ ]:
cols = ['patient_identifier1', 'patient_identifier2', 'patient_identifier3', 'resultId', 'resultDate', 'label', 'valueNum', 'unitOfMeasure']

rename_dict = {'resultDate' : 'measurement_datetime', 'label' : 'measurement_source_value_name', 'valueNum' : 'value_as_number', 'unitOfMeasure' : 'unit_source_value'}

epic_egfr_df_exploded = epic_egfr_df.explode('document_Fields').reset_index(drop=True)

epic_egfr_results_df = pd.concat(
    [
        epic_egfr_df_exploded.drop(columns=['document_Fields']),
        epic_egfr_df_exploded['document_Fields'].apply(pd.Series)
    ],
    axis=1
)

rel_tests = ["eGFR", "eGFR by CKD-EPI (2009)", "eGFR by MDRD", "EGFR BY CKD -15 MINUTES"]

epic_egfr_results_df = epic_egfr_results_df[(epic_egfr_results_df['label'].isin(rel_tests))&(epic_egfr_results_df['valueNum'].notna())&(epic_egfr_results_df['valueNum']<70)][cols].rename(columns=rename_dict).reset_index(drop=True)

del epic_egfr_df, epic_egfr_df_exploded, rel_tests, rename_dict

epic_egfr_results_df['measurement_datetime'] = pd.to_datetime(epic_egfr_results_df['measurement_datetime'])
epic_egfr_results_df['measurement_date'] = epic_egfr_results_df['measurement_datetime'].dt.date

epic_egfr_results_df['patient_identifier3'] = epic_egfr_results_df['patient_identifier3'].astype(str)
epic_egfr_results_df['patient_identifier3'] = epic_egfr_results_df['patient_identifier3'].apply(lambda x: x[:-2] if x[-2:] =='.0' else x)

In [ ]:
hospital_number_df = epic_egfr_results_df['patient_identifier1'].explode().reset_index().set_index('index')

epic_egfr_results_df = epic_egfr_results_df.drop(columns='patient_identifier1').join(hospital_number_df)

del hospital_number_df

In [ ]:
with duckdb.connect() as conn:
    epic_results_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                                       SELECT DISTINCT coalesce(nhs.master_person_id, epic.master_person_id, hosp.master_person_id) AS master_person_id
                                              , coalesce(p1.birth_datetime, p2.birth_datetime, p3.birth_datetime) AS birth_datetime
                                              , m.measurement_date
                                              , 'eGFR' AS measurement_source_value_name
                                              , m.value_as_number
                                              , 'mL/min' AS unit_source_value
                                        FROM epic_egfr_results_df as m
                                            LEFT JOIN int_person_identifier_lookup AS nhs
                                                ON nhs.patient_id = m.patient_identifier3
                                            LEFT JOIN int_person_identifier_lookup AS epic
                                                ON epic.patient_id = m.patient_identifier2
                                            LEFT JOIN int_person_identifier_lookup AS hosp
                                                ON hosp.patient_id = m.patient_identifier2
                                            LEFT JOIN ext_person as p1
                                                ON p1.master_person_id = nhs.master_person_id
                                            LEFT JOIN ext_person as p2
                                                ON p2.master_person_id = epic.master_person_id
                                            LEFT JOIN ext_person as p3
                                                ON p3.master_person_id = hosp.master_person_id
                                        WHERE nhs.master_person_id IS NOT NULL OR epic.master_person_id IS NOT NULL OR hosp.master_person_id IS NOT NULL
                                            ;""").df()

In [ ]:
measure_gfrs_df = pd.concat([epic_results_df, omop_measures_df])

measure_gfrs_df = measure_gfrs_df.sort_values(by=['master_person_id', 'measurement_date']).drop_duplicates().reset_index(drop=True)

#### Number of GFR Results <70

In [ ]:
measure_gfrs_df.shape

## Get all Measures <60
This will act at the base masurement table, with appropriate 3 & 12 Month Date Filters

In [ ]:
first_measures_df = duckdb.sql("""SELECT DISTINCT master_person_id
                                        , birth_datetime
                                        , measurement_date
                                        , measurement_source_value_name
                                        , value_as_number
                                        , unit_source_value
                                        , DATE_ADD(measurement_date, INTERVAL 3 MONTH) AS measurement_date_3monthfilter
                                        , DATE_ADD(measurement_date, INTERVAL 12 MONTH) AS measurement_date_12monthfilter
                                    FROM measure_gfrs_df
                                    WHERE value_as_number < 60
                                    ORDER BY master_person_id, measurement_date""").df()

In [ ]:
first_measures_df.shape

In [ ]:
first_measures_df['master_person_id'].nunique()

## Get all Measurements <60 Occuring Between 3 & 12 Months after the First Result

In [ ]:
second_measure_df = duckdb.sql("""SELECT DISTINCT t1.master_person_id
                                         , t1.birth_datetime
                                         , t1.measurement_date
                                         , t1.measurement_source_value_name
                                         , t1.value_as_number
                                         , t1.unit_source_value
                                         , t2.measurement_date AS measurement_date_second
                                         , t2.value_as_number AS value_as_number_second
                                         , t2.unit_source_value AS unit_source_value_second
                                         , DATE_ADD(t2.measurement_date, INTERVAL 12 MONTH) AS measurement_date_second_12monthfilter
                                    FROM first_measures_df AS t1
                                        INNER JOIN measure_gfrs_df AS t2
                                            ON t1.master_person_id = t2.master_person_id AND t2.measurement_date >= measurement_date_3monthfilter
                                                                                         AND t2.measurement_date <= measurement_date_12monthfilter
                                    WHERE t2.value_as_number < 60
                                    ORDER BY t1.master_person_id, t1.measurement_date, t2.measurement_date""").df()

In [ ]:
del first_measures_df

second_measure_df.shape

Number of Unique Patients with an initial GFR Result <60 and a subsequent Result <60 between 3 & 12 Months after the first, thus meeting the First & Second Inclusion Criteria

In [ ]:
second_measure_df['master_person_id'].nunique()

To Reduce unnecessary duplication of rows, only the earliest unique pars of First and Second Measurment Dates are retained.

This also helps speed up processing and reduce computational load

In [ ]:
base_measure_df = second_measure_df.groupby(['master_person_id', 'measurement_date'])['measurement_date_second'].min().reset_index()
base_measure_df.columns = ['master_person_id', 'measurement_date', 'measurement_date_second_min']

base_measure_df.head()

In [ ]:
second_measure_refined_df = second_measure_df.merge(base_measure_df, how="inner", on=['master_person_id', 'measurement_date'])

second_measure_refined_df.head()

In [ ]:
second_measure_refined_df.shape

In [ ]:
second_measure_refined_df = second_measure_df.merge(base_measure_df, how="inner", on=['master_person_id', 'measurement_date'])

second_measure_refined_df = second_measure_refined_df[second_measure_refined_df['measurement_date_second']==second_measure_refined_df['measurement_date_second_min']].drop(columns='measurement_date_second_min')

del base_measure_df, second_measure_df

second_measure_refined_df.shape

## Get All Measurements <70 Occuring at least 12 Months After the Second Result

In [ ]:
patients_of_interest_df = duckdb.sql("""SELECT DISTINCT t1.master_person_id
                                               , t1.birth_datetime
                                               , t1.measurement_date
                                               , t1.measurement_source_value_name
                                               , t1.value_as_number
                                               , t1.unit_source_value
                                               , t1.measurement_date_second
                                               , t1.value_as_number_second
                                               , t1.unit_source_value_second
                                               , t2.measurement_date AS measurement_date_final
                                               , t2.value_as_number AS value_as_number_final
                                               , t2.unit_source_value AS unit_source_value_final
                                        FROM second_measure_refined_df AS t1
                                            INNER JOIN measure_gfrs_df AS t2
                                                ON t1.master_person_id = t2.master_person_id AND t2.measurement_date >= measurement_date_second_12monthfilter
                                        WHERE t2.value_as_number < 70
                                        ORDER BY t1.master_person_id, t1.measurement_date, t1.measurement_date_second, t2.measurement_date""").df()

In [ ]:
patients_of_interest_df.head()

In [ ]:
del second_measure_furthur_refined_df, measure_gfrs_df

patients_of_interest_df.shape

Much like above, reducing unnecessary duplication of rows

In [ ]:
base_measure_df = patients_of_interest_df.groupby(['master_person_id', 'measurement_date', 'measurement_date_second'])['measurement_date_final'].min().reset_index()
base_measure_df.columns = ['master_person_id', 'measurement_date', 'measurement_date_second', 'measurement_date_final_min']

base_measure_df.head()

In [ ]:
patients_of_interest_refined_df = patients_of_interest_df.merge(base_measure_df, how="inner", on=['master_person_id', 'measurement_date', 'measurement_date_second'])

patients_of_interest_refined_df = patients_of_interest_refined_df[patients_of_interest_refined_df['measurement_date_final']==patients_of_interest_refined_df['measurement_date_final_min']].drop(columns='measurement_date_final_min')

patients_of_interest_refined_df = patients_of_interest_refined_df.groupby(['master_person_id', 'birth_datetime'])['measurement_date'].first().reset_index()

patients_of_interest_refined_df.shape

In [ ]:
patients_of_interest_refined_df.head() 

## Filter for Adult Patients Only

In [ ]:
inclusion_patients_df = duckdb.sql("""SELECT DISTINCT master_person_id
                                             , birth_datetime AS dateOfBirth
                                             , ROUND((CAST(measurement_date AS DATE)-CAST(birth_datetime AS DATE))/365.2425, 2) AS ageAtFirstMeasure
                                             , measurement_date AS measureDate
                                      FROM patients_of_interest_refined_df
                                      WHERE (CAST(measurement_date AS DATE)-CAST(birth_datetime AS DATE))/365.2425 > 18 --OR (CAST(measurement_date_second AS DATE)-CAST(birth_datetime AS DATE))/365.2425 > 18
                                      ORDER BY master_person_id, measureDate""").df()

In [ ]:
inclusion_patients_df.shape

In [ ]:
inclusion_patients_df.head()

## Get First Results for Each Patients
This is the earliest stage each paitient will be included based on criteria set out

In [ ]:
refined_df = inclusion_patients_df.groupby('master_person_id').first().reset_index()

In [ ]:
del patients_of_interest_refined_df, inclusion_patients_df

refined_df.shape

## Add in Patient Identifiers, and Patient Details

In [ ]:
with duckdb.connect() as conn:
    full_inclusion_pats_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                                       SELECT DISTINCT m.master_person_id
                                              , nhs.patient_id AS patient_identifier3
                                              , gstt.patient_id AS patient_identifier4
                                              , epic.patient_id as patient_identifier2
                                              , m.dateOfBirth
                                              , d.death_date as deathOfDate
                                              , p.gender_concept_Name AS gender
                                              , p.race_concept_code AS raceCode
                                              , p.race_concept_name AS race
                                              , m.ageAtFirstMeasure
                                              , m.measureDate
                                        FROM refined_df as m
                                            LEFT JOIN (SELECT master_person_id, patient_id FROM int_person_identifier_lookup WHERE patient_id_type = 'nhs_number') AS nhs
                                                USING(master_person_id)
                                            LEFT JOIN (SELECT master_person_id, patient_id FROM int_person_identifier_lookup WHERE patient_id_type = 'gstt_local_id') AS gstt
                                                USING(master_person_id)
                                            LEFT JOIN (SELECT master_person_id, patient_id FROM int_person_identifier_lookup WHERE patient_id_type = 'epic_mrn') AS epic
                                                USING(master_person_id)
                                            LEFT JOIN ext_person AS p
                                                USING(master_person_id)
                                            LEFT JOIN ext_death AS d
                                                USING(master_person_id)
                                        ORDER BY m.master_person_id, m.measureDate;""").df()

In [ ]:
full_inclusion_pats_df.shape

Due to the issue with each unique patient have potentially numerous different identifiers, group identifiers so that each row corresponds to a unique patient

In [ ]:
agg = {
    'patient_identifier3': set,
    'patient_identifier4': set,
    'patient_identifier2': set
}

In [ ]:
pat_ids_df = full_inclusion_pats_df[['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2']]
pat_ids_df = pat_ids_df.groupby('master_person_id').aggregate(agg).reset_index()

del agg, refined_df

In [ ]:
cols = full_inclusion_pats_df.columns.to_list()

full_inclusion_pats_clean_df = full_inclusion_pats_df.drop(columns=['patient_identifier3', 'patient_identifier4', 'patient_identifier2']).drop_duplicates() 

full_inclusion_pats_clean_df = full_inclusion_pats_clean_df.merge(pat_ids_df, how='left', on='master_person_id')
full_inclusion_pats_clean_df = full_inclusion_pats_clean_df[cols]

del full_inclusion_pats_df, pat_ids_df, cols

full_inclusion_pats_clean_df.head()

In [ ]:
full_inclusion_pats_clean_df.shape

In [ ]:
with duckdb.connect() as conn:
    full_inclusion_pats_final_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                                       SELECT DISTINCT m.master_person_id
                                              , m.patient_identifier3
                                              , m.patient_identifier4
                                              , m.patient_identifier2
                                              , m.dateOfBirth
                                              , m.deathOfDate
                                              , m.gender
                                              , m.raceCode
                                              , m.race
                                              , d.postcode
                                              , d.imd_decile
                                              , d.imd_rank
                                              , m.ageAtFirstMeasure
                                              , m.measureDate
                                        FROM full_inclusion_pats_clean_df as m
                                            LEFT JOIN (SELECT DISTINCT pd.master_person_id
                                                              , pd.postcode
                                                              , pd.imd_decile
                                                              , pd.imd_rank
                                                              , pd.valid_from
                                                              , max.max_valid_from
                                                       FROM int_person_deprivation AS pd
                                                           LEFT JOIN (SELECT DISTINCT master_person_id
                                                                             , MAX(valid_from) AS max_valid_from
                                                                      FROM int_person_deprivation
                                                                      GROUP BY master_person_id) AS max
                                                                USING (master_person_id)
                                                       WHERE pd.valid_from = max.max_valid_from AND pd.valid_to IS NULL) AS d
                                                ON m.master_person_id = d.master_person_id
                                        ORDER BY m.master_person_id, m.measureDate;""").df()

In [ ]:
full_inclusion_pats_final_df.head()

In [ ]:
full_inclusion_pats_final_df.shape

In [ ]:
full_inclusion_pats_final_df['master_person_id'].nunique()

### Flag Additional Patients with EPIC Search Results

In [ ]:
initial_inclusion_pats_df = pd.read_csv(data_path+"raw_data/elasticsearch_search_hits/chronic_kidney_disease_inclusion_patients.csv")['master_person_id']

In [ ]:
initial_pats_list = list(set(initial_inclusion_pats_df.to_list()))

In [ ]:
full_inclusion_pats_final_df['addl_pat_flg'] = full_inclusion_pats_final_df['master_person_id'].apply(lambda x: 0 if x in initial_pats_list else 1)

In [ ]:
full_inclusion_pats_final_df.head()

## Export Data

In [ ]:
data_path = "../data/raw_data/elasticsearch_search_hits/"

In [ ]:
full_inclusion_pats_final_df.to_csv(data_path+"chronic_kidney_disease_inclusion_patients.csv", index=False)

## Sandbox